# prepare DRG scRNAseq reference to label Xenium data

In this notebook, we combine two DRG scRNAseq references, one containing neurons and one containing non-neurons. Filtering and embedding is done on both these references in `/notebooks/00b_download_references`
- GSE139088-scvi-leiden.h5ad processed from https://www.ncbi.nlm.nih.gov/geo/query/acc.cgi?acc=GSE139088
- GSE254789-nonneurons.h5ad processed from https://www.ncbi.nlm.nih.gov/geo/query/acc.cgi?acc=GSE254789

**Pinned Environment:** [`envs/sc-scvi.yaml`](../../envs/sc-scvi.yaml)  

In [ ]:
from pathlib import Path
import os
import scanpy as sc
from scipy.sparse import issparse
import scvi
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from collections import Counter
import anndata as ad
from lightning.pytorch import seed_everything
import random
import torch
import sys
import session_info

In [ ]:
random.seed(0)
seed_everything(0)

scvi.settings.seed = 0
scvi.settings.num_workers = 32

In [ ]:
sys.path.append(str(Path.cwd().resolve().parents[1]))
from config.paths import BASE_DIR
print('BASE_DIR:', BASE_DIR)

# input ref data
input_dir = BASE_DIR / "data" / "scrna-seq" / "h5ad" / "04_clustered"

scvi_dir = BASE_DIR / 'scvi' / 'models'

output_dir = BASE_DIR / "data" / "scrna-seq" / "h5ad" / "05_reference"
output_dir.mkdir(parents=True, exist_ok=True)

In [ ]:
adata_1 = sc.read_h5ad(os.path.join(input_dir, "GSE139088-scvi-leiden.h5ad"))
adata_2 = sc.read_h5ad(os.path.join(input_dir, "GSE254789-nonneurons.h5ad"))

adata_list = [adata_1, adata_2]

In [ ]:
for data in adata_list:
    data.X = data.layers['counts'].copy()

In [ ]:
adata = ad.concat(adata_list, 
                  join="outer", 
                  label = 'batch', # suffix of adata.obs_names should match batch
                  index_unique="-")

In [ ]:
adata.obs["cell_label"] = np.where(
    adata.obs["batch"].astype(int) == 0,
    adata.obs["original_annotation"].astype(str).values,
    adata.obs["cell_type"].astype(str).values
)

In [ ]:
adata.obs.cell_label.value_counts()

## Prepare adata

In [ ]:
adata.X[:5, :5].toarray()

In [ ]:
# Save counts layer for scVI and downstream processing
adata.X = adata.layers["counts"].copy()

print('adata.X is sparse:', issparse(adata.X))
print('adata.X has only whole numbers:', np.all(adata.X.data == np.round(adata.X.data)))  # True if all values are whole numbers

In [ ]:
adata.X[:5, :5].toarray()

## scVI

#### Train model

In [ ]:
# Set up AnnData for SCVI
scvi.model.SCVI.setup_anndata(
    adata, 
    layer="counts",  # Use raw count data
    batch_key = 'batch'
)

# Initialize model
model = scvi.model.SCVI(
    adata,
    gene_likelihood = "zinb" # zero inflated to account for different modalities
)

# Train SCVI with default settings
print('training model')
model.train(accelerator = 'gpu', early_stopping = True)

# save model
model.save(scvi_dir, prefix='GSE139088_GSE254789_')

# add scVI latent dimensions to adata.obsm
adata.obsm['X_scVI'] = model.get_latent_representation(adata).astype(np.float32)
adata.obsm['X_scVI'].shape

# run umap
sc.pp.neighbors(adata, use_rep = 'X_scVI', random_state = 0)
sc.tl.umap(adata, random_state = 0)

# save adata
adata.obs["sample_number"] = adata.obs["sample_number"].astype(str)
filename = os.path.join(output_dir, 'adata-reference.h5ad') # saved with raw layer active
print(filename)
os.makedirs(os.path.dirname(filename), exist_ok=True)

adata.write_h5ad(filename, compression='gzip')



In [ ]:
# plot umap
sc.pl.umap(adata, color=['total_counts', 'cell_label'], legend_fontsize=10, frameon=False)


# plot training 
plt.plot(model.history["elbo_train"], label="Train ELBO Loss")
plt.plot(model.history["elbo_validation"], label="Validation ELBO Loss")
plt.xlabel("Epoch")
plt.ylabel("Loss")
plt.legend()
plt.title("SCVI Training Loss Curve")
plt.show()